# Feature Selection

We'll use the insights gained from our visualizations to create an appropriate feature set.

### Common Imports

In [ ]:
import pandas as pd
import pickle as pkl
import cloudpickle as cpkl

from sklearn.model_selection import train_test_split

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

In [19]:
df = pkl.load(open("../../../data/processed/vbac/filtered_df.pkl", "rb"))

Based on findings from our visualizations, we will use the following features:

In [20]:
feature_set = [
    "augmentation_of_labor",
    "induction_of_labor",
    "attendant_at_birth",
    "prior_births_now_living",
    "number_of_previous_cesarean",
    "BMI",
    "interval_since_last_live_birth",
    "combined_gestation_detail_in_weeks",
    "successful_vbac",
]

In [21]:
filtered_df = df[feature_set].copy()

filtered_df.describe()

,augmentation_of_labor,induction_of_labor,attendant_at_birth,prior_births_now_living,number_of_previous_cesarean,BMI,interval_since_last_live_birth,combined_gestation_detail_in_weeks,successful_vbac
count,553932.000000,553932.000000,553932.000000,553932.000000,553932.000000,553932.000000,553932.000000,553932.000000,553932.000000
mean,0.061934,0.069927,1.187007,1.854933,1.535616,30.870369,88.841316,38.191596,0.141149
std,0.241035,0.255025,0.569777,2.489849,3.366269,11.913619,188.683497,2.671544,0.348176
min,0.000000,0.000000,1.000000,0.000000,1.000000,13.000000,3.000000,17.000000,0.000000
25%,0.000000,0.000000,1.000000,1.000000,1.000000,24.100000,25.000000,37.000000,0.000000
50%,0.000000,0.000000,1.000000,1.000000,1.000000,28.300000,41.000000,39.000000,0.000000
75%,0.000000,0.000000,1.000000,2.000000,2.000000,34.200000,71.000000,39.000000,0.000000
max,1.000000,1.000000,9.000000,99.000000,99.000000,99.900000,999.000000,99.000000,1.000000


In [22]:
filtered_df.info()

<class 'pandas.DataFrame'>
Index: 553932 entries, 11 to 3669916
Data columns (total 9 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   augmentation_of_labor               553932 non-null  float64
 1   induction_of_labor                  553932 non-null  float64
 2   attendant_at_birth                  553932 non-null  int64  
 3   prior_births_now_living             553932 non-null  int64  
 4   number_of_previous_cesarean         553932 non-null  int64  
 5   BMI                                 553932 non-null  float64
 6   interval_since_last_live_birth      553932 non-null  int64  
 7   combined_gestation_detail_in_weeks  553932 non-null  int64  
 8   successful_vbac                     553932 non-null  int64  
dtypes: float64(3), int64(6)
memory usage: 42.3 MB


Our dataset does not include any nulls, but it does include some "unknown" values encoded as 9, 99, or 999. We'll change those to be nulls and re-evaluate.

In [23]:
filtered_df["attendant_at_birth"] = filtered_df["attendant_at_birth"].replace(
    {9: pd.NA}
)
filtered_df["prior_births_now_living"] = filtered_df["prior_births_now_living"].replace(
    {99: pd.NA}
)
filtered_df["number_of_previous_cesarean"] = filtered_df[
    "number_of_previous_cesarean"
].replace({99: pd.NA})
filtered_df["BMI"] = filtered_df["BMI"].replace({99: pd.NA})
filtered_df["interval_since_last_live_birth"] = filtered_df[
    "interval_since_last_live_birth"
].replace({999: pd.NA})
filtered_df["combined_gestation_detail_in_weeks"] = filtered_df[
    "combined_gestation_detail_in_weeks"
].replace({99: pd.NA})

In [24]:
filtered_df.info()

<class 'pandas.DataFrame'>
Index: 553932 entries, 11 to 3669916
Data columns (total 9 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   augmentation_of_labor               553932 non-null  float64
 1   induction_of_labor                  553932 non-null  float64
 2   attendant_at_birth                  553779 non-null  object 
 3   prior_births_now_living             553646 non-null  object 
 4   number_of_previous_cesarean         553303 non-null  object 
 5   BMI                                 553932 non-null  float64
 6   interval_since_last_live_birth      532713 non-null  object 
 7   combined_gestation_detail_in_weeks  553763 non-null  object 
 8   successful_vbac                     553932 non-null  int64  
dtypes: float64(3), int64(1), object(5)
memory usage: 42.3+ MB


Our null count is insignificant, so we'll opt to just drop our nulls.

In [25]:
filtered_df = filtered_df.dropna()

In [26]:
filtered_df.info()

<class 'pandas.DataFrame'>
Index: 531835 entries, 11 to 3669913
Data columns (total 9 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   augmentation_of_labor               531835 non-null  float64
 1   induction_of_labor                  531835 non-null  float64
 2   attendant_at_birth                  531835 non-null  object 
 3   prior_births_now_living             531835 non-null  object 
 4   number_of_previous_cesarean         531835 non-null  object 
 5   BMI                                 531835 non-null  float64
 6   interval_since_last_live_birth      531835 non-null  object 
 7   combined_gestation_detail_in_weeks  531835 non-null  object 
 8   successful_vbac                     531835 non-null  int64  
dtypes: float64(3), int64(1), object(5)
memory usage: 40.6+ MB


In [27]:
successful_vbac = filtered_df["successful_vbac"]
filtered_df = filtered_df.drop(columns=["successful_vbac"])

In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    filtered_df,
    successful_vbac,
    train_size=0.8,
    random_state=14,
    stratify=successful_vbac,
)

In [29]:
cat_features = [
    "augmentation_of_labor",
    "induction_of_labor",
    "attendant_at_birth",
]

cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")
)

In [30]:
num_features = [
    "prior_births_now_living",
    "number_of_previous_cesarean",
    "BMI",
    "interval_since_last_live_birth",
    "combined_gestation_detail_in_weeks",
]

num_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())

In [31]:
class PriorBirthPreviousCesareanTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_transformed = X.copy()
        X_transformed["prior_births*number_of_previous_cesarean"] = (
            X_transformed["prior_births_now_living"]
            * X_transformed["number_of_previous_cesarean"]
        ).astype(int)
        X_transformed = X_transformed.drop(
            columns=["prior_births_now_living", "number_of_previous_cesarean"]
        )
        return X_transformed

In [32]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, num_features),
        ("cat", cat_pipeline, cat_features),
        (
            "custom",
            PriorBirthPreviousCesareanTransformer(),
            ["prior_births_now_living", "number_of_previous_cesarean"],
        ),
    ]
)

In [33]:
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

In [34]:
cpkl.dump(preprocessor, open("../../../models/vbac/preprocessor.pkl", "wb"))

pkl.dump(
    X_train_prepared, open("../../../data/processed/vbac/X_train_prepared.pkl", "wb")
)
pkl.dump(
    X_test_prepared, open("../../../data/processed/vbac/X_test_prepared.pkl", "wb")
)
pkl.dump(y_train, open("../../../data/processed/vbac/y_train.pkl", "wb"))
pkl.dump(y_test, open("../../../data/processed/vbac/y_test.pkl", "wb"))